In [0]:
from pyspark.sql import functions as F
from functools import reduce

BASE_PATH = "/Volumes/workspace/default/cic_ids2017/raw"
DELTA_PATH = "/Volumes/workspace/default/cic_ids2017/processed/cic_ids2017_delta"

# Discover all CSV files
files = sorted([
    f.path for f in dbutils.fs.ls(BASE_PATH)
    if f.name.lower().endswith(".csv")
])

dataframes = []

for path in files:

    filename = path.split("/")[-1]

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(path)
    )

    # Normalize column names
    df = df.toDF(*[
        c.strip().replace("\ufeff", "")
        for c in df.columns
    ])

    # Normalize labels
    df = df.withColumn(
        "Label",
        F.trim(F.col("Label"))
    )

    # Preserve source information
    df = df.withColumn(
        "_source_file",
        F.lit(filename)
    )

    dataframes.append(df)

# Combine datasets
cic_df = reduce(
    lambda x, y: x.unionByName(
        y, allowMissingColumns=True
    ),
    dataframes
)

# Save as Delta
cic_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.columnMapping.mode", "name") \
    .save(DELTA_PATH)

print("Delta dataset saved successfully!")
print("Rows:", f"{cic_df.count():,}")
print("Location:", DELTA_PATH)

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Selected numerical features
numeric_features = [
    "Destination Port",
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Flow IAT Mean",
    "Packet Length Mean",
    "Packet Length Std",
    "SYN Flag Count",
    "FIN Flag Count"
]

# Verify column availability
available_features = [
    c for c in numeric_features
    if c in cic_df.columns
]

missing_features = [
    c for c in numeric_features
    if c not in cic_df.columns
]

print("Available features:", len(available_features))
print("Missing features:", missing_features)

# Create an analysis-only DataFrame
analysis_df = cic_df.select(
    "Label",
    *available_features
)

# Binary traffic category
analysis_df = analysis_df.withColumn(
    "binary_label",
    F.when(
        F.upper(F.trim(F.col("Label"))) == "BENIGN",
        "BENIGN"
    ).otherwise("ATTACK")
)

# Convert selected features to numerical values.
# NaN and infinity are replaced with null ONLY in this analysis copy.
for i, feature in enumerate(available_features):

    raw = F.col(f"`{feature}`").cast("double")

    cleaned = F.when(
        F.isnan(raw) | (F.abs(raw) == float("inf")),
        None
    ).otherwise(raw)

    analysis_df = analysis_df.withColumn(
        f"feature_{i}",
        cleaned
    )

print("Numerical analysis dataset prepared.")

In [0]:
# Build one aggregation across all selected features
agg_expressions = []

for i, feature in enumerate(available_features):

    col = F.col(f"feature_{i}")

    agg_expressions.extend([
        F.count(col).alias(f"count_{i}"),
        F.avg(col).alias(f"mean_{i}"),
        F.stddev(col).alias(f"std_{i}"),
        F.min(col).alias(f"min_{i}"),
        F.percentile_approx(
            col, 0.50, 10000
        ).alias(f"median_{i}"),
        F.percentile_approx(
            col, 0.95, 10000
        ).alias(f"p95_{i}"),
        F.max(col).alias(f"max_{i}")
    ])

# Aggregate by binary traffic category
statistics_wide = (
    analysis_df
    .groupBy("binary_label")
    .agg(*agg_expressions)
)

# Only two aggregated rows are transferred to Pandas
stats_pdf = statistics_wide.toPandas()

# Convert to a readable long-format table
records = []

for _, row in stats_pdf.iterrows():

    for i, feature in enumerate(available_features):

        records.append({
            "Feature": feature,
            "Traffic Class": row["binary_label"],
            "Valid Count": row[f"count_{i}"],
            "Mean": row[f"mean_{i}"],
            "Std": row[f"std_{i}"],
            "Min": row[f"min_{i}"],
            "Median": row[f"median_{i}"],
            "P95": row[f"p95_{i}"],
            "Max": row[f"max_{i}"]
        })

feature_statistics = pd.DataFrame(records)

display(feature_statistics)

In [0]:
import matplotlib.pyplot as plt
import numpy as np

# Sampled pandas copy of the analysis dataset for visualization
sample_pdf = (
    analysis_df
    .sample(withReplacement=False, fraction=0.1, seed=42)
    .toPandas()
)

important_features = [
    "Flow Duration",
    "Flow IAT Mean",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Packet Length Mean",
    "Packet Length Std",
    "Total Fwd Packets",
    "Total Backward Packets",
    "SYN Flag Count",
    "FIN Flag Count"
]

for feature in important_features:

    feature_index = available_features.index(feature)
    feature_col = f"feature_{feature_index}"

    benign = sample_pdf.loc[
        sample_pdf["binary_label"] == "BENIGN",
        feature_col
    ].dropna()

    attack = sample_pdf.loc[
        sample_pdf["binary_label"] == "ATTACK",
        feature_col
    ].dropna()

    benign = benign[np.isfinite(benign)]
    attack = attack[np.isfinite(attack)]

    # Signed logarithmic transformation for visualization
    benign = np.sign(benign) * np.log1p(np.abs(benign))
    attack = np.sign(attack) * np.log1p(np.abs(attack))

    fig, ax = plt.subplots(figsize=(9, 5))

    ax.boxplot(
        [benign, attack],
        tick_labels=["BENIGN", "ATTACK"],
        showfliers=False
    )

    ax.set_title(
        f"Distribution Comparison: {feature}",
        fontsize=14
    )

    ax.set_ylabel("Signed Log1p Value")
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

In [0]:
from pyspark.sql import functions as F
from functools import reduce

negative_features = [
    "Flow Duration",
    "Flow Bytes/s",
    "Flow Packets/s"
]

negative_results = []

for feature in negative_features:

    value = F.col(f"`{feature}`").cast("double")

    result = (
        cic_df
        .select(
            "Label",
            "_source_file",
            value.alias("feature_value")
        )
        .filter(
            F.col("feature_value").isNotNull() &
            (F.col("feature_value") < 0)
        )
        .groupBy("Label", "_source_file")
        .agg(
            F.count("*").alias("Negative Records")
        )
        .withColumn(
            "Feature",
            F.lit(feature)
        )
    )

    negative_results.append(result)

negative_summary = reduce(
    lambda x, y: x.unionByName(y),
    negative_results
)

display(
    negative_summary.orderBy(
        "Feature",
        F.desc("Negative Records")
    )
)

In [0]:
from pyspark.sql import functions as F

# Convert the three features to numerical values
duration = F.col("`Flow Duration`").cast("double")
bytes_rate = F.col("`Flow Bytes/s`").cast("double")
packets_rate = F.col("`Flow Packets/s`").cast("double")

# Define negative-value conditions
negative_duration = duration < 0
negative_bytes = bytes_rate < 0
negative_packets = packets_rate < 0

# Calculate intersections
overlap_analysis = cic_df.agg(
    F.count(
        F.when(negative_duration, 1)
    ).alias("Negative Duration"),

    F.count(
        F.when(negative_packets, 1)
    ).alias("Negative Packets/s"),

    F.count(
        F.when(negative_bytes, 1)
    ).alias("Negative Bytes/s"),

    F.count(
        F.when(negative_duration & negative_packets, 1)
    ).alias("Duration AND Packets/s"),

    F.count(
        F.when(negative_duration & negative_bytes, 1)
    ).alias("Duration AND Bytes/s"),

    F.count(
        F.when(
            negative_duration &
            negative_packets &
            negative_bytes,
            1
        )
    ).alias("All Three Features"),

    F.count(
        F.when(
            negative_duration |
            negative_packets |
            negative_bytes,
            1
        )
    ).alias("Any Negative Feature")
)

display(overlap_analysis)

In [0]:
%pip install seaborn

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Exclude Destination Port because its numeric value
# represents a port identifier, not a continuous measurement.
correlation_features = [
    f for f in available_features
    if f != "Destination Port"
]

# Select the sample already created for visualization
corr_columns = [
    f"feature_{available_features.index(f)}"
    for f in correlation_features
]

corr_data = sample_pdf[corr_columns].copy()

# Replace infinite values and remove missing observations
corr_data = corr_data.replace(
    [float("inf"), float("-inf")],
    float("nan")
)

# Calculate Pearson correlations
correlation_matrix = corr_data.corr(method="pearson")

# Rename columns for readability
correlation_matrix.columns = correlation_features
correlation_matrix.index = correlation_features

# Visualization
fig, ax = plt.subplots(figsize=(15, 12))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=ax
)

ax.set_title(
    "CIC-IDS2017 – Numerical Feature Correlation Matrix",
    fontsize=16,
    pad=20
)

plt.xticks(rotation=60, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()